In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import os
import csv

import os

os.environ['TF_NUM_INTPAOP_THREADS'] = '4'
os.environ['TF_NUM_INTEROP_THREADS'] = '4'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

In [34]:
E_matrix = 2600.0
E_fiber = 230000.0
strength_matrix = 70.0
strength_fiber = 3500.0

In [35]:
SHAPE = (100, 100, 100)
BASE_PATH_RVES = "./rves"
BASE_PATH_CURVES = "./displacement-force"
CURVE_FILES = [f"displacement-force{i}.csv" for i in range(1, 46)]
STRAIN_GRID = None
N_STRAIN_POINTS = 101
L0_MM = 1.0 
AREA_MM2 = 1.0
FIBER_RADIUS = 3.5
FIBER_LENGTH = SHAPE[2]
RNG = np.random.default_rng()

In [36]:
def read_rve_from_folder(folder_path, shape):
    rve = np.zeros(shape, dtype=np.float32)
    base_name = os.path.basename(folder_path)
    for z in range(shape[2]):
        layer_file = os.path.join(folder_path, f"{base_name}_layer_{z+1}.csv")
        if not os.path.exists(layer_file):
            layer_file = os.path.join(folder_path, f"layer_{z+1}.csv")
        if not os.path.exists(layer_file):
            raise FileNotFoundError(f"Не найден файл слоя {z+1} в {folder_path}")
        with open(layer_file, 'r') as f:
            reader = csv.reader(f)
            layer_data = []
            for row in reader:
                layer_data.append([int(float(x)) for x in row])
            layer = np.array(layer_data, dtype=np.float32)
            rve[:, :, z] = layer
    return rve

In [37]:
def read_curve(csv_path):
    try:
        data = np.loadtxt(csv_path, delimiter=',', skiprows=1, encoding='utf-8-sig')
    except ValueError:
        data = np.loadtxt(csv_path, delimiter=';', skiprows=1, encoding='utf-8-sig')
    
    if data.ndim == 1:
        data = data.reshape(-1, 2)
    displacement = data[:, 0]
    force = data[:, 1]
    strain = displacement / L0_MM
    stress = force / AREA_MM2
    return strain, stress

In [38]:
def resample_curve(strain, stress, target_strain_grid):
    f = interp1d(strain, stress, kind='linear', fill_value=(stress[0], stress[-1]), bounds_error=False)
    stress_resampled = f(target_strain_grid)
    stress_resampled = np.maximum(stress_resampled, 0)
    return stress_resampled

In [39]:
def load_all_data(base_path_rves, base_path_curves, num_samples=12, curve_files=None, target_strain_grid=None):
    X_list = []
    Y_list = []
    strain_grid = None
    
    if curve_files is None:
        curve_files = [f"displacement-force{i}.csv" for i in range(1, num_samples+1)]

    if target_strain_grid is None:
        all_strains = []
        for i in range(num_samples):
            curve_path = os.path.join(base_path_curves, curve_files[i])
            if not os.path.exists(curve_path):
                print(f"Предупреждение: файл {curve_path} не найден, пропускаем.")
                continue
            strain, _ = read_curve(curve_path)
            all_strains.append(strain)
        if not all_strains:
            raise FileNotFoundError("Не найдено ни одного файла кривых для построения сетки.")
        max_strain = max([np.max(s) for s in all_strains])
        strain_grid = np.linspace(0, max_strain, N_STRAIN_POINTS)
    else:
        strain_grid = target_strain_grid
    
    for i in range(num_samples):
        rve_folder = os.path.join(base_path_rves, f"rve_{i+1}")
        if not os.path.isdir(rve_folder):
            print(f"Папка {rve_folder} не найдена, пропускаем.")
            continue
        
        curve_path = os.path.join(base_path_curves, curve_files[i])
        if not os.path.exists(curve_path):
            print(f"Кривая {curve_path} не найдена, пропускаем RVE {i+1}.")
            continue
        
        try:
            rve = read_rve_from_folder(rve_folder, shape=SHAPE)
        except Exception as e:
            print(f"Ошибка чтения RVE {i+1}: {e}")
            continue
        
        strain, stress = read_curve(curve_path)
        stress_resampled = resample_curve(strain, stress, strain_grid)
        
        X_list.append(rve)
        Y_list.append(stress_resampled)
        print(f"Загружен образец {i+1}: RVE shape {rve.shape}, кривая {len(stress_resampled)} точек")
    
    if not X_list:
        raise RuntimeError("Не удалось загрузить ни одного образца.")
    
    X = np.array(X_list)[..., np.newaxis]
    Y = np.array(Y_list)
    return X, Y, strain_grid

In [40]:
def build_3d_cnn(input_shape, output_dim):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv3D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPool3D(2)(x)
    x = layers.Conv3D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPool3D(2)(x)
    x = layers.Conv3D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPool3D(2)(x)
    x = layers.Conv3D(256, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    outputs = layers.Dense(output_dim, activation='linear')(x)
    model = models.Model(inputs, outputs)
    return model

In [ ]:
def augment_data(X, Y):
    X_aug = []
    Y_aug = []
    for i in range(X.shape[0]):
        X_aug.append(X[i])
        Y_aug.append(Y[i])
        X_aug.append(np.flip(X[i], axis=0))
        Y_aug.append(Y[i])
        X_aug.append(np.flip(X[i], axis=1))
        Y_aug.append(Y[i])
    return np.array(X_aug), np.array(Y_aug)

In [42]:
from scipy.interpolate import interp1d

X, y, strain_grid = load_all_data(BASE_PATH_RVES, BASE_PATH_CURVES, 
                                  num_samples=45, curve_files=CURVE_FILES, 
                                  target_strain_grid=None)
if X.shape[0] < 2:
    print("Недостаточно данных")
    exit()

X_aug, y_aug = augment_data(X, y)

X_train, X_val, y_train, y_val = train_test_split(X_aug, y_aug, 
                                                  test_size=0.1111111, 
                                                  random_state=42)

max_stress = np.max(y_aug)
y_train_norm = y_train / max_stress
y_val_norm = y_val / max_stress

Загружен образец 1: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 2: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 3: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 4: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 5: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 6: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 7: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 8: RVE shape (100, 100, 100), кривая 101 точек
Ошибка чтения RVE 9: Не найден файл слоя 1 в ./rves/rve_9
Папка ./rves/rve_10 не найдена, пропускаем.
Папка ./rves/rve_11 не найдена, пропускаем.
Загружен образец 12: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 13: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 14: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 15: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 16: RVE shape (100, 100, 100), кривая 101 точек
Загружен образец 

In [43]:
model = build_3d_cnn(input_shape=(*SHAPE, 1), output_dim=N_STRAIN_POINTS)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

history = model.fit(X_train, y_train_norm, 
                    epochs=30,
                    batch_size=1, 
                    validation_data=(X_val, y_val_norm),
                    verbose=1)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 100, 100, 100,  │             0 │
│                                 │ 1)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_8 (Conv3D)               │ (None, 100, 100, 100,  │           896 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_6 (MaxPooling3D)  │ (None, 50, 50, 50, 32) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_9 (Conv3D)               │ (None, 50, 50, 50, 64) │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_7 (MaxPooling3D)  │ (None, 25, 25, 25, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_10 (Conv3D)              │ (None, 25, 25, 25,     │       221,312 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_8 (MaxPooling3D)  │ (None, 12, 12, 12,     │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_11 (Conv3D)              │ (None, 12, 12, 12,     │       884,992 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling3d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling3D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 101)            │        25,957 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,451,429 (5.54 MB)

 Trainable params: 1,451,429 (5.54 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - loss: 0.0338 - mae: 0.1226 - val_loss: 0.0028 - val_mae: 0.0427
Epoch 2/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 140s 1s/step - loss: 0.0079 - mae: 0.0651 - val_loss: 0.0011 - val_mae: 0.0268
Epoch 3/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - loss: 0.0088 - mae: 0.0729 - val_loss: 0.0049 - val_mae: 0.0506
Epoch 4/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - loss: 0.0048 - mae: 0.0522 - val_loss: 0.0017 - val_mae: 0.0295
Epoch 5/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - loss: 0.0038 - mae: 0.0455 - val_loss: 0.0011 - val_mae: 0.0264
Epoch 6/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - loss: 0.0021 - mae: 0.0344 - val_loss: 0.0035 - val_mae: 0.0495
Epoch 7/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - loss: 0.0023 - mae: 0.0362 - val_loss: 0.0022 - val_mae: 0.0371
Epoch 8/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - loss: 0.0023 - mae: 0.0353 - val_loss: 0.0011 - val_mae: 0.0276
Epoch 9/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/

In [44]:
y_pred_norm = model.predict(X_val)
y_pred = y_pred_norm * max_stress
y_val_true = y_val

W0000 00:00:1777551716.192399   10880 cpu_allocator_impl.cc:82] Allocation of 1792000000 exceeds 10% of free system memory.


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [45]:
mse = mean_squared_error(y_val_true.flatten(), y_pred.flatten())
mae = mean_absolute_error(y_val_true.flatten(), y_pred.flatten())
r2 = r2_score(y_val_true.flatten(), y_pred.flatten())
print(f"\nМетрики на валидации:")
print(f"MSE = {mse:.4f}")
print(f"MAE = {mae:.4f} МПа")
print(f"R²  = {r2:.4f}")


Метрики на валидации:
MSE = 32684.5288
MAE = 130.2511 МПа
R²  = 0.9953


In [46]:
strength_true = np.max(y_val_true, axis=1)
strength_pred = np.max(y_pred, axis=1)
rel_error = np.abs(strength_true - strength_pred) / strength_true
mean_rel_error = np.mean(rel_error) * 100
print(f"Средняя относительная ошибка прочности: {mean_rel_error:.2f}%")

Средняя относительная ошибка прочности: 3.16%


In [2]:
y_pred_norm = model.predict(X_val)
y_val_true = y_val_norm * max_stress
y_pred = y_pred_norm * max_stress

n_plots = min(5, len(X_val))
if n_plots > 0:
    idx = np.random.choice(len(X_val), size=n_plots, replace=False)
else:
    idx = []

plt.figure(figsize=(12, 8))
for i, id in enumerate(idx):
    plt.subplot(2, 3, i+1)
    plt.plot(strain_grid, y_val_true[id], 'b-', label='True')
    plt.plot(strain_grid, y_pred[id], 'r--', label='Pred')
    plt.xlabel('Strain')
    plt.ylabel('Stress (MPa)')
    plt.legend()
    plt.title(f'Sample {id}')
plt.tight_layout()
plt.savefig('real_data_comparison.png')
plt.show()

NameError: name 'model' is not defined

In [48]:
model.save('cnn_real_data.h5')